# Experiment 2 layer-controller diagnostics

Inspect quality-gated layer alpha trajectories, intervention decisions, AdamW alignment, direct control-probe effects, cooldowns, and global pauses. Set `NANOGPT_EXPERIMENT2_SCALE` to `level0` or `level1`.

In [ ]:
import json, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SCALE=os.getenv('NANOGPT_EXPERIMENT2_SCALE','level0')
STATE=Path(os.getenv('NANOGPT_EXPERIMENT2_STATE_FILE',f'/tmp/nanogpt-experiment2-{SCALE}-current-pair'))
PAIR=Path(os.getenv('NANOGPT_EXPERIMENT2_PAIR_ROOT',STATE.read_text().strip()))
ROOT=PAIR/'adaptive/results'
print('scale:',SCALE); print('pair:',PAIR)
runs=sorted(ROOT.glob('adamw_adaptive_wwpgd_seed_*'))
assert runs, f'No adaptive runs under {ROOT}'

def read_all(filename):
    frames=[]
    for run in runs:
        path=run/filename
        if not path.exists(): continue
        d=pd.read_csv(path); d['seed']=int(run.name.rsplit('_',1)[1]); d['run']=str(run); frames.append(d)
    return pd.concat(frames,ignore_index=True) if frames else pd.DataFrame()

layers=read_all('layer_measurements.csv')
events=read_all('projection_events.csv')
windows=read_all('controller_windows.csv')
print('seeds:',sorted(layers.seed.unique()) if len(layers) else [])
print('layer rows:',len(layers),'projection rows:',len(events),'window rows:',len(windows))
assert len(layers)


In [ ]:
layers['alpha']=pd.to_numeric(layers.alpha,errors='coerce')
fig,ax=plt.subplots(figsize=(16,8))
for (seed,name),g in layers.dropna(subset=['alpha']).groupby(['seed','layer_name']):
    ax.plot(g.step,g.alpha,linewidth=1,alpha=.65,label=f'{seed}:{name}')
ax.axhline(2.0,linestyle='--',linewidth=1.5,label='target alpha = 2')
ax.axhspan(1.9,2.1,alpha=.08,label='exit band')
ax.axhspan(1.7,2.3,alpha=.04,label='entry band')
ax.set(xlabel='optimizer step',ylabel='WeightWatcher alpha',title=f'Experiment 2 {SCALE}: all matrix alpha trajectories')
ax.grid(alpha=.2)
ax.legend(loc='upper left',bbox_to_anchor=(1.01,1),fontsize=6,ncol=2)
plt.tight_layout(); plt.show()


In [ ]:
valid=layers.dropna(subset=['alpha']).copy()
valid['in_exit_band']=(valid.alpha-2).abs()<=.10
valid['in_entry_band']=(valid.alpha-2).abs()<=.30
occupancy=valid.groupby(['seed','step']).agg(measured=('alpha','size'),in_exit_band=('in_exit_band','sum'),in_entry_band=('in_entry_band','sum'),median_alpha=('alpha','median')).reset_index()
occupancy['exit_fraction']=occupancy.in_exit_band/occupancy.measured
occupancy['entry_fraction']=occupancy.in_entry_band/occupancy.measured
fig,ax=plt.subplots(figsize=(12,5))
for seed,g in occupancy.groupby('seed'): ax.plot(g.step,g.exit_fraction,label=f'seed {seed}')
ax.set(xlabel='optimizer step',ylabel='fraction within alpha 2 ± 0.10',title=f'Experiment 2 {SCALE}: target-band occupancy')
ax.set_ylim(0,1.02); ax.grid(alpha=.25); ax.legend(); plt.show()
display(occupancy.groupby('seed').tail(1))


In [ ]:
if len(events):
    for c in ['alignment_cosine','projection_to_adamw_ratio_requested','probe_loss_delta','relative_frobenius_change_applied','alpha_improvement']:
        if c in events: events[c]=pd.to_numeric(events[c],errors='coerce')
    display(events.groupby(['seed','projection_status']).size().rename('rows').reset_index())
    accepted=events[events.projection_status=='projected'].copy()
    fig,ax=plt.subplots(figsize=(10,6))
    if len(accepted):
        ax.scatter(accepted.alignment_cosine,accepted.probe_loss_delta,s=12,alpha=.45)
    ax.axhline(0,linestyle='--',linewidth=1); ax.axvline(0,linestyle='--',linewidth=1)
    ax.set(xlabel='cosine(AdamW update, WWPGD correction)',ylabel='isolated control-probe loss change',title='Accepted candidate alignment versus immediate probe effect')
    ax.grid(alpha=.25); plt.show()
else:
    print('No projection events are available yet.')


In [ ]:
if len(events):
    layer_effects=events.groupby(['seed','layer_name']).agg(attempts=('projection_status','size'),accepted=('changed','sum'),mean_probe_delta=('probe_loss_delta','mean'),mean_alignment=('alignment_cosine','mean'),mean_alpha_improvement=('alpha_improvement','mean'),mean_applied_change=('relative_frobenius_change_applied','mean')).reset_index()
    display(layer_effects.sort_values(['seed','mean_probe_delta']).head(100))


In [ ]:
if len(windows):
    for c in ['progress_advantage','loss_gap','global_pause_until']:
        if c in windows: windows[c]=pd.to_numeric(windows[c],errors='coerce')
    fig,ax=plt.subplots(figsize=(12,5))
    for seed,g in windows.groupby('seed'): ax.plot(g.step,g.progress_advantage,label=f'seed {seed}')
    ax.axhline(0,linestyle='--',linewidth=1)
    ax.set(xlabel='optimizer step',ylabel='adaptive progress − baseline progress',title='Reference-guided window advantage')
    ax.grid(alpha=.25); ax.legend(); plt.show()
    display(windows.tail(30))
